# Variables declaration and stopword detection

In [1]:
%load_ext autoreload
%autoreload 2

from Extractor import Extractor
from StopWordsDetector import StopwordDetector
from helper_functions import *

dir_path = "./data/corpus2mw/"
#dir_path = "./data/corpusRecall2/"
special_chars = [',',':',';','.','!','?','[',']','(',')','<','>','#','|','=','{','}']
vowels_and_accented_vowels = ["a","e","i","o","u","A","E","I","O","U",
                        "à","á","ã","â","é","è","í","ó","õ","ò",
                        "Á","À","Ã","Â","Õ","Ô","ê","Ê","y","Y"]  # Set of vowels for syllable counting
vowels = ["a","e","i","o","u","A","E","I","O","U","y","Y"] # Set of accented vowels
corpus = load_and_preprocess_corpus(dir_path, special_chars)

extractor = Extractor(corpus, n_max=7, limit=None, glue_type="phi_square")

stopword_detector = StopwordDetector(extractor, 
                                     vowels_and_accented_vowels=vowels_and_accented_vowels, 
                                     vowels=vowels)
stopwords = stopword_detector.detect_stopwords()
# delete the StopwordDetector object to free memory
del stopword_detector

Finding n-grams: 100%|██████████| 7/7 [00:32<00:00,  4.58s/it]


### Glue values retrieval and frequency filtering

In [2]:
extractor.find_glue_values()

# Optional --> Filter out by min freq
extractor.filter_by_min_frequency(2)

Calculating glue values:   0%|          | 0/10461429 [00:00<?, ?n-gram/s]

Calculating glue values: 100%|██████████| 10461429/10461429 [01:11<00:00, 145581.74n-gram/s]


### Parallel implementation

In [3]:
MWEs = extractor.find_MWEs(stopwords=stopwords, parallel=True)
print("MWEs found:")
print(extractor.MWEs)
parallel_MWEs = extractor.MWEs

Using 7 workers for parallel processing.
Building super-ngram index...
Preparing relevant n-grams per chunk...


Finding MWEs: 100%|██████████| 462068/462068 [00:00<00:00, 882589.14n-gram/s] 

MWEs found:
[('year', '2014'), ('21st', 'year'), ('martial', 'arts'), ('aviation', 'industry'), ('entire', 'estate'), ('earn', 'promotion'), ('magnetic', 'tape'), ('potentially', 'allowing'), ('NC', 'State'), ('NCAA', 'tournament'), ('five', 'consecutive'), ('consecutive', 'years'), ('five', 'years'), ('conference', 'record'), ('defending', 'champion'), ('Sweet', 'Sixteen'), ('top', '50'), ('professional', 'golf'), ('golf', 'tournament'), ('took', 'place'), ('Golf', 'Course'), ('Town', 'Hall'), (')"', '>'), ('former', 'chairman'), ('Reverend', 'William'), ('June', '1873'), ('Christ', 'Church'), ('May', '1951'), ('Gothic', 'building'), ('LA-CO', 'Industries'), ('temperature', 'indicating'), ('farm', 'located'), ('16', 'km'), ('Atlantic', 'Ocean'), ('migratory', 'bird'), ('gastropod', 'mollusks'), ('Elementary', 'School'), ('Chuckey', 'Charles'), ('novels', 'written'), ('recording', 'label'), ('Music', 'Publishing'), ('While', 'traveling'), ('Middle', 'East'), ('"Billboard"', 'charts'), 

### Sequential implementation

In [4]:
MWEs = extractor.find_MWEs(stopwords=stopwords)
print("MWEs found:")
print(extractor.MWEs)
sequential_MWEs = extractor.MWEs

Finding MWEs: 100%|██████████| 462068/462068 [00:00<00:00, 982578.06n-gram/s] 

MWEs found:
[('year', '2014'), ('21st', 'year'), ('martial', 'arts'), ('aviation', 'industry'), ('entire', 'estate'), ('earn', 'promotion'), ('magnetic', 'tape'), ('potentially', 'allowing'), ('NC', 'State'), ('NCAA', 'tournament'), ('five', 'consecutive'), ('consecutive', 'years'), ('five', 'years'), ('conference', 'record'), ('defending', 'champion'), ('Sweet', 'Sixteen'), ('top', '50'), ('professional', 'golf'), ('golf', 'tournament'), ('took', 'place'), ('Golf', 'Course'), ('Town', 'Hall'), (')"', '>'), ('former', 'chairman'), ('Reverend', 'William'), ('June', '1873'), ('Christ', 'Church'), ('May', '1951'), ('Gothic', 'building'), ('LA-CO', 'Industries'), ('temperature', 'indicating'), ('farm', 'located'), ('16', 'km'), ('Atlantic', 'Ocean'), ('migratory', 'bird'), ('gastropod', 'mollusks'), ('Elementary', 'School'), ('Chuckey', 'Charles'), ('novels', 'written'), ('recording', 'label'), ('Music', 'Publishing'), ('While', 'traveling'), ('Middle', 'East'), ('"Billboard"', 'charts'), 

In [5]:
# checking the difference between parallel and sequential MWEs to assert that parallel extraction is working correctly

print(set(parallel_MWEs) - set(sequential_MWEs))

set()


## Precision and recall evaluation

In [6]:
def compute_recall(valid_mwes, predicted_mwes):
    valid_set = set(valid_mwes)
    predicted_set = set(predicted_mwes)

    # True Positives: valid MWEs that are found by the extractor
    true_positives = valid_set.intersection(predicted_set)

    tp_count = len(true_positives)
    total_valid = len(valid_mwes)

    recall = tp_count / total_valid if total_valid > 0 else 0.0

    print(f"True Positives: {tp_count}")
    print(f"Total Valid MWEs: {total_valid}")
    print(f"Recall: {recall:.3f}")

    return recall, true_positives

gold_mwes = [
    ('John', 'Philoponus'),
    ('Most', 'modern'),
    ('body', 'parts'),
    ('19th', 'century'),
    ('civil', 'war'),
    ('Indic', 'scripts'),
    ('Southeast', 'Asia'),
    ('a', 'monastery'),
    ('Catholic', 'Church'),
    ('Orthodox', 'Church'),
    ('World', 'War'),
    ('2004', 'Summer'),
    ('UEFA', 'Cup'),
    ('United', 'States'),
    ('adverse', 'events'),
    ('New', 'Orleans'),
    ('Holy', 'Land'),
    ('three', 'times'),
    ('a', 'letter'),
    ('Latin', 'Empire'),
    ('second', 'wife'),
    ('fuel', 'prices'),
    ('crime', 'rate'),
    ('national', 'average'),
    ('population', 'pyramid'),
    ('maximum', 'transfer'),
    ('E-mail', 'advertising'),
    ('John', 'Lee'),
    ('Lee', 'Hooker'),
    ('Summer', 'Paralympics'),
    ('small', 'enough'),
    ('Elizabeth', 'I'),
    ('World', 'War', 'II'),
    ('murder', 'of', 'Naboth'),
    ('Peter', 'of', 'Courtenay'),
    ('second', 'and', 'third'),
    ('Louis', 'the', 'Child'),
    ('megabytes', 'per', 'second'),
    ('a', 'hydrogen', 'ion'),
    ('produces', 'its', '"conjugate'),
    ('John', 'Lee', 'Hooker'),
    ('enough', 'to', 'fit'),
    ('head', 'of', 'a', 'monastery'),
    ('below', 'the', 'national', 'average'),
    ('removal', 'of', 'a', 'hydrogen'),
    ('the', 'head', 'of', 'a', 'monastery'),
    ('depart', 'for', 'the', 'Holy', 'Land'),
    ('removal', 'of', 'a', 'hydrogen', 'ion'),
    ('Odyssey', 'Arena'),
    ('Belfast', 'Giants'),
    ('Matroid', 'theory'),
    ('common', 'law'),
    ('Czech', 'Republic'),
    ('big', 'bang'),
    ('Turkic', 'languages'),
    ('life', 'expectancy'),
    ('October', '1974'),
    ('1974', 'election.'),
    ('general', 'election'),
    ('the', 'Conservatives'),
    ('first', 'time'),
    ('real', 'humans'),
    ('first', 'appearance'),
    ('football', 'game'),
    ('Coronae', 'Borealis'),
    ('double', 'star'),
    ('giant', 'star'),
    ('Sigma', 'Coronae'),
    ('October', '1974', 'election.'),
    ('the', 'first', 'time'),
    ('the', 'Common', 'Lisp'),
    ('the', 'first', 'appearance'),
    ('orbit', 'each', 'other'),
    ('light-years', 'from', 'Earth'),
    ('star', 'of', 'magnitude'),
    ('Sigma', 'Coronae', 'Borealis'),
    ('dog', 'meat'),
    ('hip', 'hop'),
    ('Mesha', 'Stele'),
    ('floating', 'point'),
    ('per', 'second'),
    ('double', 'bass'),
    ('Canary', 'Wharf'),
    ('New', 'York'),
    ('the', 'Alpha'),
    ('Alpha', 'processors'),
    ('two', 'digits'),
    ('May', '1919'),
    ('encyclopedia', 'articles'),
    ('dictionary', 'entries'),
    ('European', 'Council'),
    ('biomass', 'production'),
    ('migratory', 'species'),
    ('Historical', 'Society'),
    ('civil', 'conflicts'),
    ('floating', 'point', 'operations'),
    ('family', 'names'),
    ('Aung', 'San'),
    ('the', 'daughter'),
    ('Ne', 'Win'),
    ('First', 'Law'),
    ('constant', 'velocity'),
    ('net', 'force'),
    ('formal', 'languages'),
    ("Newton's", 'First', 'Law'),
    ('Mexico', 'City'),
    ('the', 'country'),
    ('extended', 'format'),
    ('McKellen', 'appeared'),
    ('intercalary', 'years'),
    ('Holyland', 'Tower'),
    ('Roman', 'Empire'),
    ('Asian', 'Avars'),
    ('4th', 'century'),
    ('Grand', 'Master'),
    ('Kent', 'State'),
    ('Pei', '&', 'Partners'),
    ('England', 'and', 'Wales'),
    ('average', 'energy'),
    ('military', 'budget'),
    ('trillion', 'rubles'),
    ("Russia's", 'military'),
    ('West', 'Virginia'),
    ('community', 'organizations'),
    ('Stadio', 'Olimpico'),
    ('20th', 'century'),
    ('Soviet', 'Union'),
    ('meat', 'dishes'),
    ('Sikh', 'religion'),
    ('Akal', 'Takht'),
    ('Stock', 'Exchange'),
    ('steam', 'engine'),
    ('Count', 'of', 'Flanders'),
    ('New', 'York', 'Times"'),
    ('city', 'of', 'Rome'),
    ('regular', 'season'),
    ('"milk', 'chocolate"'),
    ('Middle', 'East'),
    ('North', 'America'),
    ('South', 'Australian'),
    ('New', 'Zealand'),
    ('married', 'couples'),
    ('living', 'together'),
    ('Census', 'Bureau'),
    ('maternal', 'blood'),
    ('reflection', 'coefficient'),
    ('21st', 'century'),
    ('Muslim-majority', 'countries'),
    ('Cambridge', 'Circus'),
    ('Economic', 'Activities'),
    ('confectionery', 'manufacturing'),
    ('Forest', 'Service'),
    ('Park', 'Service'),
    ('North', 'Cascades'),
    ('Chairman', 'Mao'),
    ('Times', 'Square'),
    ('the', 'Middle', 'East'),
    ('during', 'World', 'War'),
    ('medical', 'treatment'),
    ('Ben', 'Hecht"'),
    ('Silver', 'Purchase', 'Act'),
    ('Senator', 'John', 'Sherman'),
    ('population', 'were', 'below', 'the', 'poverty'),
    ('Communist', 'Party'),
    ('High', 'School'),
    ('European', 'settlers'),
    ('House', 'of', 'Representatives'),
    ('indigenous', 'Maya'),
    ('South', 'Park'),
    ('Napoleonic', 'Wars'),
    ('Warner', 'Bros'),
    ('James', 'Kinney'),
    ('Gotti', 'Jr'),
    ('Holy', 'Roman', 'Emperor'),
    ('early', '1950s'),
    ('Second', 'Sophistic'),
    ('Maria', 'Theresa'),
    ('first', 'order', 'conditions'),
    ('continuum', 'hypothesis'),
    ('economic', 'activities'),
    ('theological', 'determinism'),
    ('physical', 'determinism'),
    ('MTV', 'Video', 'Music', 'Awards'),
    ('Grammy', 'Awards'),
    ('Cypress', 'Hill'),
    ('GZA', 'appeared'),
    ('"Soul', 'Assassins'),
    ('DJ', 'Muggs'),
    ('West', 'Berlin'),
    ('Norman', 'French'),
    ('Viet', 'Minh'),
    ('March', '1946'),
    ('French', 'Union'),
    ('Tarot', 'de', 'Marseille'),
    ('Local', 'Government', 'Act'),
    ('Olympic', 'Games'),
    ('National', 'Register'),
    ('Rhein-Neckar', 'Triangle'),
    ('Julian', 'calendar'),
    ('higher', 'energy')
]

print(len(gold_mwes))
compute_recall(gold_mwes, sequential_MWEs)

200
True Positives: 94
Total Valid MWEs: 200
Recall: 0.470


(0.47,
 {('"Soul', 'Assassins'),
  ('"milk', 'chocolate"'),
  ('19th', 'century'),
  ('20th', 'century'),
  ('21st', 'century'),
  ('Akal', 'Takht'),
  ('Asian', 'Avars'),
  ('Aung', 'San'),
  ('Cambridge', 'Circus'),
  ('Canary', 'Wharf'),
  ('Catholic', 'Church'),
  ('Communist', 'Party'),
  ('Coronae', 'Borealis'),
  ('Count', 'of', 'Flanders'),
  ('Cypress', 'Hill'),
  ('Czech', 'Republic'),
  ('DJ', 'Muggs'),
  ('Economic', 'Activities'),
  ('European', 'Council'),
  ('Gotti', 'Jr'),
  ('Grand', 'Master'),
  ('High', 'School'),
  ('Historical', 'Society'),
  ('Holyland', 'Tower'),
  ('House', 'of', 'Representatives'),
  ('Indic', 'scripts'),
  ('MTV', 'Video', 'Music', 'Awards'),
  ('Maria', 'Theresa'),
  ('Matroid', 'theory'),
  ('May', '1919'),
  ('McKellen', 'appeared'),
  ('Mesha', 'Stele'),
  ('Middle', 'East'),
  ('Most', 'modern'),
  ('Muslim-majority', 'countries'),
  ('Napoleonic', 'Wars'),
  ('National', 'Register'),
  ('Ne', 'Win'),
  ("Newton's", 'First', 'Law'),
  ('O

In [7]:
import random
# precision
sampled_mwes = random.sample(extractor.MWEs, 200)

print(sampled_mwes)

#PRECISION AND RECALL
scp_p = 177 / 200
print(f"SCP Precision = {scp_p:.2f}")

dice_p = 168 / 200
print(f"Dice Precision = {dice_p:.2f}")

phys_p = 170 / 200
print(f"Phys Squared Precision = {phys_p:.2f}")

[('Ph', '.', 'D'), ('much', 'changed'), ('Harris', 'wrote'), ('manufacturing', 'plant'), ('von', 'Kirchbach'), ('securely', 'attached', 'individuals'), ('Procter', '&', 'Gamble'), ('television', 'and', 'radio'), ('Nazi', 'German'), ('Ellenberger', 'Park'), ("Weebl's", 'Stuff'), ('houses', 'were', 'built'), ('"Creatures', 'of', 'the', 'Night"'), ('Just', 'north'), ('Headquarters', 'U', '.S', '.', 'Air'), ('protoconch', 'are', 'smooth'), (':', 'Note'), ('musician', 'and', 'producer'), ('Comando', 'Mega'), ('Buffalo', 'Jump'), ('.5', 'meters'), ('inscribed', 'stone'), ('weather', 'conditions'), ('become', 'famous'), ('Sacrifice', 'of', 'Isaac'), ('headed', 'by', 'former'), ('Congregation', 'for', 'the', 'Doctrine'), ('since', '2008'), ('text', 'is', 'written', 'in', 'one', 'column'), ('given', 'by', 'Gilgamesh'), ('18', 'year'), ('James', 'Madison'), ('television', 'season'), (':', 'Many'), ('race', 'record'), ('weak', 'point'), ('Lakhta', 'Center'), ('Women', '18-49'), ('>', 'Application

## Explicit and implicit keywords retrieval

In [8]:

# Extract explicit keywords
top_n_explicit = 15
explicit_keywords = extractor.extract_explicit_keywords(top_n=top_n_explicit)

print("\nExplicit Keywords:")
for i, kw in enumerate(explicit_keywords):
    print(f"{i + 1}: {' '.join(kw)}")

# Extract implicit keywords based on similarity
top_n_implicit = 10
implicit_keywords = extractor.extract_implicit_keywords(explicit_keywords, top_n=top_n_implicit)

print("\nImplicit Keywords (based on similarity to explicit):")
for i, (kw, score) in enumerate(implicit_keywords):
    print(f"{i + 1}: {kw} (similarity: {score:.4f})")

# TODO - Complete with true_keywords = trueLabels/Gold Standard
""" true_keywords = [...]  # <- No clue what the gold standard is.

predicted_keywords = [' '.join(k) if isinstance(k, tuple) else k for k in explicit_keywords]
predicted_keywords += [kw for kw, _ in implicit_keywords]

precision, recall, f1 = extractor.evaluate_keywords(predicted_keywords, true_keywords)
print("\nEvaluation Metrics:")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}") """



Explicit Keywords:
1: year 2014
2: 21st year
3: martial arts
4: aviation industry
5: entire estate
6: earn promotion
7: magnetic tape
8: potentially allowing
9: NC State
10: NCAA tournament
11: five consecutive
12: consecutive years
13: five years
14: conference record
15: defending champion

Implicit Keywords (based on similarity to explicit):
1: year 2014 (similarity: 1.0000)
2: aviation industry (similarity: 1.0000)
3: earn promotion (similarity: 1.0000)
4: NCAA tournament (similarity: 1.0000)
5: 21st year (similarity: 1.0000)
6: martial arts (similarity: 1.0000)
7: entire estate (similarity: 1.0000)
8: magnetic tape (similarity: 1.0000)
9: potentially allowing (similarity: 1.0000)
10: NC State (similarity: 1.0000)


' true_keywords = [...]  # <- No clue what the gold standard is.\n\npredicted_keywords = [\' \'.join(k) if isinstance(k, tuple) else k for k in explicit_keywords]\npredicted_keywords += [kw for kw, _ in implicit_keywords]\n\nprecision, recall, f1 = extractor.evaluate_keywords(predicted_keywords, true_keywords)\nprint("\nEvaluation Metrics:")\nprint(f"Precision: {precision:.4f}")\nprint(f"Recall:    {recall:.4f}")\nprint(f"F1 Score:  {f1:.4f}") '